# Training Neural Networks (MLP) with PyTorch Lightning

In [2]:
import os
import pickle
import polars as pl
import numpy as np
import pandas as pd
import torch
device = torch.device(f'cuda:2' if torch.cuda.is_available() else 'cpu')
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'

# Load Data

In [3]:
df = pl.scan_parquet('train.parquet').collect().to_pandas()
valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
feature_names = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)]
label_name = 'responder_6'
X_train = df[ feature_names ]
y_train = df[ label_name ]
w_train = df[ "weight" ]
X_valid = valid[ feature_names ]
y_valid = valid[ label_name ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

((21022056, 88),
 (21022056,),
 (21022056,),
 (1082224, 88),
 (1082224,),
 (1082224,))

## Select Features with MA window size 7

In [4]:
# 1) Define helper function
def add_moving_averages(df, columns_to_transform, date_col='date_id', window=7, min_periods=1):
    for col in columns_to_transform:
        new_col_name = f"{col}_ma{window}"
        df[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
        )
    return df

# 3) Select which features to compute MAs for
features_for_ma = [f"feature_{i:02d}" for i in range(79)]
df = add_moving_averages(df, features_for_ma)
valid = add_moving_averages(valid, features_for_ma)

# 4) Decide if you want these new MA features for your model
use_ma_features = True

# Original feature names
feature_names = [f"feature_{i:02d}" for i in range(79)] + \
                [f"responder_{idx}_lag_1" for idx in range(9)]

if use_ma_features:
    ma_feature_names = [f"{col}_ma7" for col in features_for_ma]
    feature_names += ma_feature_names

# 5) Set label & weight
label_name = 'responder_6'
weight_name = 'weight'

# Train grand XGB on all 79 + 79MA = 166 features, and find the most important ones

In [5]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

feature_names = [f"feature_{i:02d}" for i in range(79)]  # original features
feature_names += [f"{col}_ma7" for col in feature_names]  # hypothetical MA features
label_name = "responder_6"
weight_name = "weight"

# Extract data from the dataframe
X = df[feature_names]
y = df[label_name]
if weight_name in df.columns:
    w = df[weight_name]

model = XGBRegressor(
    tree_method='gpu_hist',    # Build trees on the GPU
    device = "cuda",          # Use the GPU
    predictor='gpu_predictor', # Do inference on the GPU
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective="reg:squarederror"
)
print(X.shape, y.shape)

# Also ensure sample_weight has the same length as y
print(w.shape if w is not None else "No sample weights")
print(X.dtypes)

# Fit the model (with sample_weight if you have one)
if w is not None:
    model.fit(X, y, sample_weight=w)
else:
    model.fit(X, y)

# -------------------------------------
# 3) Extract feature importances
# -------------------------------------
importances = model.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

# Sort features by importance descending
importance_df.sort_values("importance", ascending=False, inplace=True)
importance_df.reset_index(drop=True, inplace=True)

importance_df




(21022056, 158) (21022056,)
(21022056,)
feature_00        float32
feature_01        float32
feature_02        float32
feature_03        float32
feature_04        float32
                   ...   
feature_74_ma7    float64
feature_75_ma7    float64
feature_76_ma7    float64
feature_77_ma7    float64
feature_78_ma7    float64
Length: 158, dtype: object


/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [20:27:11] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [20:27:11] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [20:28:05] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


,feature,importance
0,feature_75,0.024410
1,feature_06,0.022709
2,feature_36,0.020625
3,feature_61,0.020074
4,feature_10,0.019418
...,...,...
153,feature_03,0.001705
154,feature_00,0.001536
155,feature_61_ma7,0.000000
156,feature_43_ma7,0.000000


In [6]:
importance_df[:25]

,feature,importance
0,feature_75,0.024410
1,feature_06,0.022709
2,feature_36,0.020625
3,feature_61,0.020074
4,feature_10,0.019418
5,feature_42_ma7,0.018569
6,feature_23_ma7,0.017826
7,feature_26_ma7,0.016089
8,feature_76_ma7,0.015391
9,feature_09,0.013951


In [7]:
ma_feature_names = importance_df[:50]["feature"].tolist()
top_ma_features = [col for col in ma_feature_names if col.endswith("ma7")]
top_ma_features

['feature_42_ma7',
 'feature_23_ma7',
 'feature_26_ma7',
 'feature_76_ma7',
 'feature_20_ma7',
 'feature_10_ma7',
 'feature_38_ma7',
 'feature_04_ma7',
 'feature_32_ma7',
 'feature_27_ma7',
 'feature_66_ma7',
 'feature_45_ma7',
 'feature_19_ma7',
 'feature_49_ma7',
 'feature_78_ma7',
 'feature_08_ma7',
 'feature_09_ma7',
 'feature_46_ma7',
 'feature_15_ma7',
 'feature_22_ma7',
 'feature_53_ma7',
 'feature_07_ma7']

## Add Features


In [8]:
# 1) Define helper function
def add_moving_averages(df, columns_to_transform, date_col='date_id', window=5, min_periods=1):
    for col in columns_to_transform:
        new_col_name = f"{col}_ma{window}"
        df[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
        )
    return df


df = add_moving_averages(df, top_ma_features)
valid = add_moving_averages(valid, top_ma_features)

use_ma_features = True

# Original feature names
feature_names = [f"feature_{i:02d}" for i in range(79)] + \
                [f"responder_{idx}_lag_1" for idx in range(9)]

# If you want the MA columns for only the top MA features
if use_ma_features:
    ma_feature_names = [f"{col}_ma5" for col in top_ma_features]
    feature_names += ma_feature_names

# 5) Set label & weight
label_name = 'responder_6'
weight_name = 'weight'

/tmp/ipykernel_2161913/2669421750.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col_name] = (
/tmp/ipykernel_2161913/2669421750.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col_name] = (
/tmp/ipykernel_2161913/2669421750.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d

In [9]:
X_train = df[ feature_names ]
y_train = df[ label_name ]
w_train = df[ "weight" ]
X_valid = valid[ feature_names ]
y_valid = valid[ label_name ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape
print(feature_names)

['feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_61', 'feature_62', 'feature_63', 'feature_64', 'feature_65', 'feature_66', 'feature_67', 'feature_68', 'feature_69', 'feature_70', 'feat

# Training Configurations

In [10]:
import os
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import (LightningDataModule, LightningModule, Trainer)
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer
from pytorch_lightning.loggers import WandbLogger
import wandb
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 2
        self.seed = 42
        self.model = 'nn'
        self.use_wandb = False
        self.project = 'js-xs-nn-with-lags'
        self.dname = "./input_df/"
        self.loader_workers = 4
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 5e-4
        self.dropouts = [0.1, 0.1]
        self.n_hidden = [512, 512, 256]
        self.patience = 25
        self.max_epochs = 1000
        self.N_fold = 5


my_args = custom_args()

/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# PyTorch Data Module Definition

In [11]:
class CustomDataset(Dataset):
    def __init__(self, df, accelerator):
        self.features = torch.FloatTensor(df[feature_names].values).to(accelerator)
        self.labels = torch.FloatTensor(df[label_name].values).to(accelerator)
        self.weights = torch.FloatTensor(df[weight_name].values).to(accelerator)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        w = self.weights[idx]
        return x, y, w


class DataModule(LightningDataModule):
    def __init__(self, train_df, batch_size, valid_df=None, accelerator='cpu'):
        super().__init__()
        self.df = train_df
        self.batch_size = batch_size
        self.dates = self.df['date_id'].unique()
        self.accelerator = accelerator
        self.train_dataset = None
        self.valid_df = None
        if valid_df is not None:
            self.valid_df = valid_df
        self.val_dataset = None

    def setup(self, fold=0, N_fold=5, stage=None):
        # Split dataset
        selected_dates = [date for ii, date in enumerate(self.dates) if ii % N_fold != fold]
        df_train = self.df.loc[self.df['date_id'].isin(selected_dates)]
        self.train_dataset = CustomDataset(df_train, self.accelerator)
        if self.valid_df is not None:
            df_valid = self.valid_df
            self.val_dataset = CustomDataset(df_valid, self.accelerator)

    def train_dataloader(self, n_workers=0):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=n_workers)

    def val_dataloader(self, n_workers=0):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=n_workers)
    


# NN Model Definition

In [12]:
# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            # layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1)) 
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)  

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w  #
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        if self.trainer.sanity_checking:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

# Create PyTorch Data Module

In [13]:

args = my_args

# checking device
device = torch.device(f'cuda:{args.gpuid}' if torch.cuda.is_available() and args.usegpu else 'cpu')
accelerator = 'gpu' if torch.cuda.is_available() and args.usegpu else 'cpu'
loader_device = 'cpu'


# Initialize Data Module

df[feature_names] = df[feature_names].fillna(method = 'ffill').fillna(0)
valid[feature_names] = valid[feature_names].fillna(method = 'ffill').fillna(0)
data_module = DataModule(df, batch_size=args.bs, valid_df=valid, accelerator=loader_device)

/tmp/ipykernel_2161913/680207757.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[feature_names] = df[feature_names].fillna(method = 'ffill').fillna(0)
/tmp/ipykernel_2161913/680207757.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  valid[feature_names] = valid[feature_names].fillna(method = 'ffill').fillna(0)


# Create Model and Training

In [ ]:
import gc
del df
gc.collect()
pl.seed_everything(args.seed)
for fold in range(args.N_fold):
    data_module.setup(fold, args.N_fold)
    # Obtain input dimension
    input_dim = data_module.train_dataset.features.shape[1]
    # Initialize Model
    model = NN(
        input_dim=input_dim,
        hidden_dims=args.n_hidden,
        dropouts=args.dropouts,
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    # Initialize Logger
    if args.use_wandb:
        wandb_run = wandb.init(project=args.project, config=vars(args), reinit=True)
        logger = WandbLogger(experiment=wandb_run)
    else:
        logger = None
    # Initialize Callbacks
    early_stopping = EarlyStopping('val_loss', patience=args.patience, mode='min', verbose=False)
    checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1, verbose=False, filename=f"./models/nn_{fold}.model") 
    timer = Timer()
    # Initialize Trainer
    trainer = Trainer(
        max_epochs=args.max_epochs,
        accelerator=accelerator,
        devices=[args.gpuid] if args.usegpu else None,
        logger=logger,
        callbacks=[early_stopping, checkpoint_callback, timer],
        enable_progress_bar=True,
    )
    # Start Training
    trainer.fit(model, data_module.train_dataloader(args.loader_workers), data_module.val_dataloader(args.loader_workers))
    # You can find trained best model in your local path
    print(f'Fold-{fold} Training completed in {timer.time_elapsed("train"):.2f}s')


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 453 K  | train
---------------------------------------------
453 K     Trainable params
0         Non-trainable params
453 K     Total params
1.813     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 36: 100%|██████████| 2054/2054 [01:12<00:00, 28.43it/s, v_num=1, val_r_square=0.00433]
Fold-0 Training completed in 2849.00s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 453 K  | train
---------------------------------------------
453 K     Trainable params
0         Non-trainable params
453 K     Total params
1.813     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 32: 100%|██████████| 2053/2053 [01:22<00:00, 24.98it/s, v_num=2, val_r_square=0.00523]
Fold-1 Training completed in 2620.32s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 453 K  | train
---------------------------------------------
453 K     Trainable params
0         Non-trainable params
453 K     Total params
1.813     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


...

[rank: 0] Received SIGTERM: 15


Epoch 10:  60%|█████▉    | 1230/2053 [00:44<00:29, 27.44it/s, v_num=3, val_r_square=0.00603]

SIGTERMException: 

/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


: 